# UJIIndoorLoc trên Google Colab

Notebook này tải mã từ một nhánh GitHub, tự liệt kê model và cách xử lý RSS, chạy một hoặc nhiều model rồi hiển thị bảng và biểu đồ ngay trong Colab. **Không lưu GitHub token trong notebook.** Bật GPU trước khi chạy BSRBF-KAN. Official test chỉ dùng để báo cáo sau khi chọn model bằng internal validation. Ma trận nhầm lẫn chỉ áp dụng cho dự đoán tầng của KNN/WKNN; tọa độ dùng sai số mét và CDF.


In [ ]:
import io
import zipfile
import urllib.error
import urllib.request
from pathlib import Path
import subprocess
import shutil
import sys
from google.colab import files

REPO = "VietUniverse/ips-wifi-research"
BRANCH = "main"  # đổi thành feature/data-robustness hoặc feature/kan-mixer để thử PR
root = Path("/content/repo_source")

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

if token:
    url = f"https://api.github.com/repos/{REPO}/zipball/{BRANCH}"
    try:
        request = urllib.request.Request(
            url, headers={"Authorization": f"Bearer {str(token).strip()}",
                          "Accept": "application/vnd.github+json"})
        with urllib.request.urlopen(request, timeout=120) as response:
            repo_zip = response.read()
    except urllib.error.HTTPError as error:
        print(f"GitHub trả HTTP {error.code}. Kiểm tra GITHUB_TOKEN và quyền đọc repo.")
        repo_zip = None
    except urllib.error.URLError:
        print("Không kết nối được GitHub API. Có thể dùng ZIP của repo.")
        repo_zip = None
else:
    repo_zip = None
token = None
if repo_zip is None:
    print("Upload ZIP của nhánh GitHub bạn muốn chạy (Code → Download ZIP).")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Hãy upload đúng một file ZIP của repo")
    repo_zip = next(iter(uploaded.values()))

if root.exists():
    shutil.rmtree(root)
root.mkdir()
with zipfile.ZipFile(io.BytesIO(repo_zip)) as z:
    z.extractall(root)
project_files = list(root.rglob("pyproject.toml"))
if len(project_files) != 1:
    raise RuntimeError(f"Cần đúng một pyproject.toml; tìm thấy {len(project_files)}")
project = project_files[0].parent
print("Project:", project)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(project)], check=True)
sys.path.insert(0, str(project / "src"))  # Kernel Colab dùng đúng mã vừa tải
print('Python của kernel:', sys.executable)


In [ ]:
import shutil
import urllib.request

data_dir = Path("/content/data/uji")
data_dir.mkdir(parents=True, exist_ok=True)
if not all((data_dir / name).exists() for name in ("trainingData.csv", "validationData.csv")):
    zip_path = Path("/content/ujiindoorloc.zip")
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/310/ujiindoorloc.zip", zip_path)
    extracted = Path("/content/uji_source")
    extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extracted)
    for name in ("trainingData.csv", "validationData.csv"):
        matches = list(extracted.rglob(name))
        if len(matches) != 1:
            raise RuntimeError(f"Không tìm thấy duy nhất {name}")
        shutil.copy2(matches[0], data_dir / name)
print("UJI data:", [(p.name, p.stat().st_size) for p in data_dir.glob("*.csv")])


## Chọn model và tiền xử lý

Ô sau in menu và **dừng để bạn nhập lựa chọn**. Nhập số model (có thể nhiều số, cách nhau bằng dấu phẩy), rồi chọn RSS và preset. Sau khi thành viên push file mới, đổi `BRANCH` ở ô đầu rồi chạy lại ô tải repo và ô chọn này. Smoke chỉ kiểm tra luồng; để nộp báo cáo hãy chọn preset đầy đủ. Preset Mixer chỉ chạy với `bsrbf_kan` và `rss_standard` (520 AP).


In [ ]:
import json

options = json.loads(subprocess.check_output(
    [sys.executable, "-m", "ips.experiments.cli", "--list-options"], text=True))
presets = sorted(str(p.relative_to(project)) for p in (project / "configs").glob("*.json"))
families = {"Neural": options["models"]}
if (project / "src/ips/neighbors/__init__.py").is_file():
    neighbor_probe = subprocess.run(
        [sys.executable, "-c", "import json; from ips.neighbors import available_models; print(json.dumps(sorted(available_models())))"],
        capture_output=True, text=True)
    if neighbor_probe.returncode:
        raise RuntimeError(f"Không tải được plugin KNN: {neighbor_probe.stderr}")
    families["KNN/WKNN"] = json.loads(neighbor_probe.stdout)

def choose_one(title, values):
    if not values: raise ValueError(f'Không có lựa chọn cho {title}')
    print('\n' + title)
    for i, value in enumerate(values, 1): print(f'  {i}. {value}')
    while True:
        raw = input(f'Nhập số 1..{len(values)}: ').strip()
        if raw.isdigit() and 1 <= int(raw) <= len(values): return values[int(raw)-1]
        print('Số không hợp lệ, hãy nhập lại.')

def choose_many(title, values):
    if not values: raise ValueError(f'Không có model cho {title}')
    print('\n' + title)
    for i, value in enumerate(values, 1): print(f'  {i}. {value}')
    while True:
        raw = input('Nhập số model, ví dụ 1 hoặc 1,3: ').strip()
        try:
            indices = [int(x.strip()) for x in raw.split(',')]
            if indices and len(indices) == len(set(indices)) and all(1 <= i <= len(values) for i in indices):
                return [values[i-1] for i in indices]
        except ValueError: pass
        print('Danh sách không hợp lệ, hãy nhập lại.')

selected_family = choose_one('Chọn họ model', list(families))
selected_models = choose_many('Chọn model', families[selected_family])
selected_preprocess = choose_one('Chọn cách xử lý RSS', options['preprocessors'])
family_presets = [p for p in presets if Path(p).name.startswith('knn_') == (selected_family == 'KNN/WKNN')]
selected_preset = choose_one('Chọn preset', family_presets)
preprocess_params = json.loads(input('RSS params dạng JSON (Enter để dùng {}): ').strip() or '{}')
if not isinstance(preprocess_params, dict): raise ValueError('RSS params phải là JSON object')
print('\nĐã chọn:', selected_family, selected_models, selected_preprocess, selected_preset)


In [ ]:
import torch
from datetime import datetime, timezone

RUN_NAME = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
run_dir = Path("/content/runs") / RUN_NAME
if not selected_models or not selected_preprocess or not selected_preset:
    raise ValueError("Hãy chọn model, RSS và preset")
settings = json.loads((project / selected_preset).read_text())
print('Chạy:', selected_models, '| Tiền xử lý:', selected_preprocess, '| Preset:', selected_preset)
print('Cấu hình preset:', json.dumps(settings, ensure_ascii=False, indent=2))
print('Số dòng dữ liệu:', [(p.name, sum(1 for _ in p.open()) - 1) for p in data_dir.glob('*.csv')])
from ips.experiments.visualize import render_run, compare_runs
run_dirs = []
if selected_family == "KNN/WKNN":
    settings.update(models=selected_models, preprocess=selected_preprocess,
                    preprocess_params=preprocess_params)
    selected_config = Path("/content") / f"selected_{RUN_NAME}.json"
    selected_config.write_text(json.dumps(settings), encoding="utf-8")
    subprocess.run([sys.executable, "-m", "ips.experiments.knn_cli", "--config", str(selected_config),
                    "--data-dir", str(data_dir), "--output-dir", str(run_dir)], check=True)
    render_run(run_dir)
    run_dirs.append(run_dir)
else:
    if settings.get("with_mixer") and (selected_models != ["bsrbf_kan"] or selected_preprocess != "rss_standard"):
        raise ValueError("Preset Mixer cần model bsrbf_kan và RSS rss_standard")
    print("GPU available:", torch.cuda.is_available())
    if "bsrbf_kan" in selected_models and not torch.cuda.is_available():
        print("Cảnh báo: BSRBF-KAN trên CPU có thể rất chậm.")
    for selected_model in selected_models:
        model_dir = run_dir / selected_model
        print('Đang chạy:', selected_model)
        subprocess.run([sys.executable, "-m", "ips.experiments.cli", "--config", str(project / selected_preset),
                        "--model", selected_model, "--preprocess", selected_preprocess,
                        "--preprocess-params", json.dumps(preprocess_params),
                        "--data-dir", str(data_dir), "--output-dir", str(model_dir)], check=True)
        render_run(model_dir)
        run_dirs.append(model_dir)
    if len(run_dirs) > 1:
        compare_runs(run_dirs, labels=selected_models)
print('Kết quả:', run_dir)


In [ ]:
archive = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
print("Result ZIP:", archive)
files.download(archive)
